# 🛡️ Stage-1: Insulin Event Classifier (The Watchdog)

## Problem
Our Regression model has **44% Miss Rate** - it fails to detect when insulin is needed.

## Solution: Two-Stage Approach
1. **Stage 1 (This notebook)**: Binary Classifier → "Is insulin needed?" (Yes/No)
2. **Stage 2 (Regressor)**: "How much insulin?" (only runs if Stage 1 = Yes)

## Goal
**Recall > 98%** (Miss Rate < 2%)

---

## 📦 Setup

In [ ]:
!pip install polars xgboost scikit-learn -q

from google.colab import drive
drive.mount('/content/drive')

print("✅ Setup complete")

In [ ]:
# GPU Check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
import time
import polars as pl
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    confusion_matrix, classification_report, 
    roc_auc_score, precision_recall_curve, roc_curve
)
import matplotlib.pyplot as plt
import gc

print(f"✅ Polars: {pl.__version__}")
print(f"✅ XGBoost: {xgb.__version__}")

## ⚙️ Configuration

In [ ]:
# Paths
BASE_DIR = "/content/drive/MyDrive/DiabetesKDS"
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")

TRAIN_FILE = os.path.join(DATA_DIR, "train_augmented_fixed.parquet")
VAL_FILE = os.path.join(DATA_DIR, "test_ready_for_xgboost.parquet")
MODEL_OUTPUT = os.path.join(DATA_DIR, "xgboost_insulin_classifier.json")

# Features
FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal',
    'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
]

# Encoding Maps
GENDER_MAP = {'Male': 0, 'Female': 1, 'M': 0, 'F': 1}
INSULIN_MAP = {
    'Humalog': 3, 'Novolog': 6, 'NovoLog': 6, 'Fiasp': 2,
    'Apidra': 0, 'Regular': 8, 'Humalog (Lispro)': 4
}

print("=" * 60)
print("📂 CONFIGURATION")
print("=" * 60)
for f in [TRAIN_FILE, VAL_FILE]:
    exists = os.path.exists(f)
    print(f"{'✅' if exists else '❌'} {os.path.basename(f)}")

## 📊 Step 1: Data Preparation & Binary Target

In [ ]:
# ============================================================
# PREPROCESSING FUNCTION
# ============================================================

def preprocess_df(df: pl.DataFrame) -> pl.DataFrame:
    """Apply encoding and create binary target."""
    
    # Gender encoding
    if 'gender' in df.columns and 'gender_encoded' not in df.columns:
        dtype = str(df['gender'].dtype)
        if 'String' in dtype or 'Utf8' in dtype:
            df = df.with_columns(
                pl.col('gender').map_elements(
                    lambda x: GENDER_MAP.get(x, -1) if x else -1,
                    return_dtype=pl.Int32
                ).alias('gender_encoded')
            )
        else:
            df = df.with_columns(pl.col('gender').fill_null(-1).cast(pl.Int32).alias('gender_encoded'))
    elif 'gender_encoded' not in df.columns:
        df = df.with_columns(pl.lit(-1).alias('gender_encoded'))
    
    # Insulin type encoding
    if 'insulin_type_bolus' in df.columns and 'insulin_type_encoded' not in df.columns:
        dtype = str(df['insulin_type_bolus'].dtype)
        if 'String' in dtype or 'Utf8' in dtype:
            df = df.with_columns(
                pl.col('insulin_type_bolus').map_elements(
                    lambda x: INSULIN_MAP.get(x, -1) if x else -1,
                    return_dtype=pl.Int32
                ).alias('insulin_type_encoded')
            )
        else:
            df = df.with_columns(pl.col('insulin_type_bolus').fill_null(-1).cast(pl.Int32).alias('insulin_type_encoded'))
    elif 'insulin_type_encoded' not in df.columns:
        df = df.with_columns(pl.lit(-1).alias('insulin_type_encoded'))
    
    # Impute missing columns
    for col in ['weight', 'basal', 'age', 'is_weekend']:
        if col in df.columns:
            df = df.with_columns(pl.col(col).fill_null(-1.0).cast(pl.Float32))
        else:
            df = df.with_columns(pl.lit(-1.0).cast(pl.Float32).alias(col))
    
    # ⭐ CREATE BINARY TARGET
    if 'bolus' in df.columns:
        df = df.with_columns(
            (pl.col('bolus') > 0).cast(pl.Int8).alias('target_class')
        )
    
    return df

print("✅ Preprocessing function defined")

In [ ]:
# ============================================================
# LOAD & PREPARE TRAINING DATA
# ============================================================
print("\n" + "=" * 60)
print("📊 LOADING TRAINING DATA")
print("=" * 60)

df_train = pl.read_parquet(TRAIN_FILE)
print(f"✅ Loaded: {df_train.shape[0]:,} × {df_train.shape[1]}")

df_train = preprocess_df(df_train)

# Fill missing features
for f in FEATURES:
    if f not in df_train.columns:
        df_train = df_train.with_columns(pl.lit(0.0).alias(f))

# Prepare arrays
X_train = df_train.select(FEATURES).to_numpy().astype(np.float32)
y_train = df_train['target_class'].to_numpy().astype(np.int32)

# Handle NaN
X_train = np.nan_to_num(X_train, nan=0.0)

# Calculate class imbalance
n_zeros = (y_train == 0).sum()
n_ones = (y_train == 1).sum()
ratio = float(n_zeros / n_ones) if n_ones > 0 else 1.0

print(f"\n🧮 Class Distribution:")
print(f"   Class 0 (No insulin): {n_zeros:,} ({n_zeros/len(y_train)*100:.1f}%)")
print(f"   Class 1 (Insulin):    {n_ones:,} ({n_ones/len(y_train)*100:.1f}%)")
print(f"   Imbalance Ratio:      {ratio:.2f}")

del df_train
gc.collect()

In [ ]:
# ============================================================
# LOAD & PREPARE VALIDATION DATA
# ============================================================
print("\n" + "=" * 60)
print("📊 LOADING VALIDATION DATA")
print("=" * 60)

df_val = pl.read_parquet(VAL_FILE)
print(f"✅ Loaded: {df_val.shape[0]:,} × {df_val.shape[1]}")

df_val = preprocess_df(df_val)

for f in FEATURES:
    if f not in df_val.columns:
        df_val = df_val.with_columns(pl.lit(0.0).alias(f))

X_val = df_val.select(FEATURES).to_numpy().astype(np.float32)
y_val = df_val['target_class'].to_numpy().astype(np.int32) if 'target_class' in df_val.columns else None

X_val = np.nan_to_num(X_val, nan=0.0)

if y_val is not None:
    valid_mask = ~np.isnan(y_val.astype(float))
    X_val = X_val[valid_mask]
    y_val = y_val[valid_mask]
    print(f"✅ Val prepared: {len(y_val):,} samples")
    print(f"   Class 0: {(y_val==0).sum():,}, Class 1: {(y_val==1).sum():,}")

del df_val
gc.collect()

## ⚗️ Step 2: Classifier Configuration (Recall-Optimized)

In [ ]:
# ============================================================
# XGBOOST CLASSIFIER PARAMS (AGGRESSIVE DETECTION)
# ============================================================

SCALE_POS_WEIGHT = ratio * 1.5  # Boost positive class

PARAMS = {
    # Objective: Binary Classification
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    
    # GPU
    'tree_method': 'hist',
    'device': 'cuda',
    
    # ⚡ AGGRESSIVE DETECTION SETTINGS
    'scale_pos_weight': SCALE_POS_WEIGHT,
    
    # Hyperparameters
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    
    # Regularization
    'gamma': 0.1,
    'reg_alpha': 0.5,
    'reg_lambda': 1,
    
    'seed': 42
}

NUM_ROUNDS = 500
EARLY_STOPPING = 30

print("\n" + "=" * 60)
print("⚗️ CLASSIFIER CONFIGURATION")
print("=" * 60)
print(f"   objective:        {PARAMS['objective']}")
print(f"   scale_pos_weight: {SCALE_POS_WEIGHT:.2f}")
print(f"   max_depth:        {PARAMS['max_depth']}")
print(f"   learning_rate:    {PARAMS['learning_rate']}")
print(f"   num_rounds:       {NUM_ROUNDS}")

## 🚀 Step 3: Training

In [ ]:
# ============================================================
# TRAIN CLASSIFIER
# ============================================================
print("\n" + "=" * 60)
print("🚀 TRAINING CLASSIFIER")
print("=" * 60)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=FEATURES)

evals = [(dtrain, 'train'), (dval, 'val')]
evals_result = {}

print(f"\n🏋️ Training ({NUM_ROUNDS} rounds, early_stop={EARLY_STOPPING})...")
t0 = time.time()

model = xgb.train(
    PARAMS,
    dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=evals,
    early_stopping_rounds=EARLY_STOPPING,
    evals_result=evals_result,
    verbose_eval=50
)

print(f"\n✅ Training complete in {time.time()-t0:.1f}s")
print(f"   Best iteration: {model.best_iteration}")
print(f"   Best AUC: {model.best_score:.4f}")

## 📈 Step 4: Evaluation & Optimal Threshold Search

In [ ]:
# ============================================================
# PREDICT PROBABILITIES
# ============================================================
print("\n" + "=" * 60)
print("📈 EVALUATION")
print("=" * 60)

y_prob = model.predict(dval)

# ROC-AUC
auc = roc_auc_score(y_val, y_prob)
print(f"\n   ROC-AUC: {auc:.4f}")

In [ ]:
# ============================================================
# OPTIMAL THRESHOLD SEARCH (Maximize Recall)
# ============================================================
print("\n" + "=" * 60)
print("🔍 OPTIMAL THRESHOLD SEARCH")
print("=" * 60)

thresholds = np.arange(0.05, 0.95, 0.05)
results = []

print(f"\n{'Threshold':<12} {'Recall':<10} {'Precision':<12} {'Miss Rate':<12}")
print("-" * 50)

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)
    
    # Calculate metrics
    tp = ((y_pred == 1) & (y_val == 1)).sum()
    fn = ((y_pred == 0) & (y_val == 1)).sum()
    fp = ((y_pred == 1) & (y_val == 0)).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    miss_rate = fn / (tp + fn) * 100 if (tp + fn) > 0 else 0
    
    results.append({
        'threshold': thresh,
        'recall': recall,
        'precision': precision,
        'miss_rate': miss_rate
    })
    
    marker = "⭐" if recall >= 0.98 else ""
    print(f"{thresh:<12.2f} {recall:<10.4f} {precision:<12.4f} {miss_rate:<12.2f}% {marker}")

# Find optimal threshold (Recall >= 98%)
safe_results = [r for r in results if r['recall'] >= 0.98]
if safe_results:
    # Among safe thresholds, pick highest precision
    optimal = max(safe_results, key=lambda x: x['precision'])
else:
    # Fallback to highest recall
    optimal = max(results, key=lambda x: x['recall'])

OPTIMAL_THRESHOLD = optimal['threshold']

print("\n" + "=" * 50)
print(f"🎯 OPTIMAL THRESHOLD FOR SAFETY: {OPTIMAL_THRESHOLD:.2f}")
print(f"   Recall:    {optimal['recall']*100:.2f}%")
print(f"   Precision: {optimal['precision']*100:.2f}%")
print(f"   Miss Rate: {optimal['miss_rate']:.2f}%")

In [ ]:
# ============================================================
# CONFUSION MATRIX (at Optimal Threshold)
# ============================================================
print("\n" + "=" * 60)
print(f"📊 CONFUSION MATRIX (Threshold={OPTIMAL_THRESHOLD:.2f})")
print("=" * 60)

y_pred_optimal = (y_prob >= OPTIMAL_THRESHOLD).astype(int)
cm = confusion_matrix(y_val, y_pred_optimal)

print(f"\n                    Predicted")
print(f"                 No        Yes")
print(f"         ┌─────────┬─────────┐")
print(f"Actual   │ {cm[0,0]:7,} │ {cm[0,1]:7,} │  (True Negatives / False Positives)")
print(f"  No     └─────────┼─────────┤")
print(f"         ┌─────────┼─────────┤")
print(f"Actual   │ {cm[1,0]:7,} │ {cm[1,1]:7,} │  (FALSE NEGATIVES / True Positives)")
print(f"  Yes    └─────────┴─────────┘")

false_negatives = cm[1,0]
true_positives = cm[1,1]
total_positives = false_negatives + true_positives

print(f"\n⚠️ FALSE NEGATIVES (Missed insulin events): {false_negatives:,}")
print(f"✅ TRUE POSITIVES (Correctly detected):      {true_positives:,}")
print(f"📊 Detection Rate: {true_positives/total_positives*100:.2f}%")

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================
print("\n" + "=" * 60)
print("📋 CLASSIFICATION REPORT")
print("=" * 60)

print(classification_report(y_val, y_pred_optimal, target_names=['No Insulin', 'Insulin']))

In [ ]:
# ============================================================
# VISUALIZATION: ROC & PR CURVES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_val, y_prob)
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {auc:.4f}')
axes[0].plot([0, 1], [0, 1], 'r--', linewidth=1)
axes[0].axhline(0.98, color='green', linestyle=':', label='98% Recall Target')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_val, y_prob)
axes[1].plot(recall, precision, 'g-', linewidth=2)
axes[1].axvline(0.98, color='red', linestyle=':', label='98% Recall Target')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 💾 Step 5: Save Model

In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================
print("\n" + "=" * 60)
print("💾 SAVING MODEL")
print("=" * 60)

model.save_model(MODEL_OUTPUT)
print(f"✅ Model saved: {MODEL_OUTPUT}")
print(f"   Size: {os.path.getsize(MODEL_OUTPUT)/(1024*1024):.2f} MB")

# Save optimal threshold
threshold_file = os.path.join(DATA_DIR, "classifier_threshold.txt")
with open(threshold_file, 'w') as f:
    f.write(f"{OPTIMAL_THRESHOLD}")
print(f"✅ Threshold saved: {threshold_file}")

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("🎉 STAGE-1 CLASSIFIER TRAINING COMPLETE")
print("=" * 70)

print(f"\n📊 Model Performance:")
print(f"   ROC-AUC:   {auc:.4f}")
print(f"   Threshold: {OPTIMAL_THRESHOLD:.2f}")
print(f"   Recall:    {optimal['recall']*100:.2f}%")
print(f"   Miss Rate: {optimal['miss_rate']:.2f}%")

print(f"\n🎯 Goal Achievement:")
if optimal['recall'] >= 0.98:
    print(f"   ✅ Recall > 98% ACHIEVED!")
else:
    print(f"   ⚠️ Recall < 98% - Consider adjusting parameters")

print(f"\n💾 Outputs:")
print(f"   Model: {MODEL_OUTPUT}")
print(f"   Threshold: {threshold_file}")

print("\n" + "=" * 70)